# CMU-MOSEI Preprocessing — Colab

Extracts Z_at (1536-d) and Z_v (768-d) features for all 6,277 CMU-MOSEI segments.

**Before running:**
1. Upload `segments.rar` to Google Drive (anywhere)
2. Update `DRIVE_RAR_PATH` in Cell 2 to match your Drive path
3. Runtime → Change runtime type → **T4 GPU**
4. Run all cells top to bottom

**Output:** `mosei_features.zip` saved to Drive (`DRIVE_OUTPUT_DIR`).  
Contains `z_at/*.pt` + `z_v/*.pt` — drop into `data/preprocessed/features/` locally.

**Time estimate:** ~25-35 min on T4 for 6,277 clips.

In [ ]:
# ── CONFIG — edit these paths ──────────────────────────────────────────────────
DRIVE_RAR_PATH   = "/content/drive/MyDrive/segments.rar"   # path to uploaded RAR
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/mosei_preprocess_out"  # output folder on Drive
REPO_URL         = "https://github.com/GelKyle/thesis-g10.git"    # your GitHub repo URL
REPO_BRANCH      = "exp/sarcasm-training"                         # branch to use
# ──────────────────────────────────────────────────────────────────────────────

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
print(f"Drive mounted. Output dir: {DRIVE_OUTPUT_DIR}")

## Step 2 — Install system tools + Python deps

In [ ]:
# System: unrar + ffmpeg
!apt-get update -qq && apt-get install -y -qq unrar ffmpeg
print("System tools installed.")

In [ ]:
# Python deps — only what preprocessing needs (torch/cv2/PIL/numpy already in Colab)
!pip install -q \
    openai-whisper \
    transformers \
    timm \
    insightface \
    onnxruntime-gpu \
    librosa \
    soundfile \
    torchaudio

# py-feat for AU saliency (optional — falls back to conf×sharpness if install fails)
try:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "feat"], check=True)
    print("py-feat installed — AU saliency ENABLED")
except Exception as e:
    print(f"py-feat optional install failed ({e}) — AU saliency disabled, using conf×sharpness fallback")

print("Python deps installed.")

## Step 3 — Extract segments RAR

In [ ]:
import os, subprocess

SEGMENTS_DIR = "/content/mosei_segments"
os.makedirs(SEGMENTS_DIR, exist_ok=True)

# Verify RAR exists
if not os.path.exists(DRIVE_RAR_PATH):
    raise FileNotFoundError(
        f"RAR not found at: {DRIVE_RAR_PATH}\n"
        f"Upload segments.rar to Drive and update DRIVE_RAR_PATH in Cell 1."
    )

rar_size_gb = os.path.getsize(DRIVE_RAR_PATH) / 1e9
print(f"RAR found: {DRIVE_RAR_PATH} ({rar_size_gb:.2f} GB)")
print(f"Extracting to: {SEGMENTS_DIR} ...")

result = subprocess.run(
    ["unrar", "x", "-y", DRIVE_RAR_PATH, SEGMENTS_DIR + "/"],
    capture_output=True, text=True
)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
    raise RuntimeError("unrar failed — check RAR integrity")

# Count extracted MP4s (may be in a subfolder — flatten search)
from pathlib import Path
all_mp4s = list(Path(SEGMENTS_DIR).rglob("*.mp4"))
print(f"Extracted {len(all_mp4s)} MP4 segments.")

# If unrar created a subfolder, find the actual segments dir
if all_mp4s:
    ACTUAL_SEGMENTS_DIR = str(all_mp4s[0].parent)
    print(f"Segments live at: {ACTUAL_SEGMENTS_DIR}")
else:
    raise RuntimeError("No MP4 files found after extraction — check RAR structure")

## Step 4 — Clone repo

In [ ]:
import subprocess, os

REPO_DIR = "/content/thesis"

if os.path.exists(REPO_DIR):
    print("Repo already cloned — pulling latest...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    print(f"Cloning {REPO_URL} @ {REPO_BRANCH} ...")
    subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, "--depth", "1", REPO_URL, REPO_DIR],
        check=True
    )

os.chdir(REPO_DIR)
print(f"Working dir: {os.getcwd()}")

## Step 5 — Create required directories + rebuild mosei_real.csv

In [ ]:
from pathlib import Path
import pandas as pd

# Create dirs the script expects
for d in [
    "data/preprocessed/features/z_at",
    "data/preprocessed/features/z_v",
    "data/preprocessed/audio",
    "data/preprocessed/transcripts",
    "data/processed/mosei_manifests",
]:
    Path(d).mkdir(parents=True, exist_ok=True)

# Build manifest pointing to Colab-extracted segments
seg_dir = Path(ACTUAL_SEGMENTS_DIR)
segs = sorted(seg_dir.glob("*.mp4"))

if not segs:
    raise RuntimeError(f"No MP4s found in {seg_dir} — check ACTUAL_SEGMENTS_DIR")

manifest_path = Path("data/processed/mosei_manifests/mosei_real.csv")
df = pd.DataFrame([
    {"clip_id": v.stem, "video_path": str(v.resolve())}
    for v in segs
])
df.to_csv(manifest_path, index=False)
print(f"Manifest written: {manifest_path} ({len(df)} rows)")
print(df.head(3))

## Step 6 — Verify GPU + environment

In [ ]:
import torch, sys

print(f"Python : {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA   : {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU'}")

# Quick import check
sys.path.insert(0, "/content/thesis")
try:
    from src.preprocessing.pipeline import PreprocessingPipeline
    from src.utils.config import Config
    print("Repo imports OK")
except ImportError as e:
    raise ImportError(f"Repo import failed: {e} — check REPO_URL/REPO_BRANCH in Cell 1")

try:
    import whisper
    print("whisper OK")
except ImportError:
    raise ImportError("openai-whisper not installed — re-run Step 2")

try:
    import insightface
    print("insightface OK")
except ImportError:
    print("WARNING: insightface not installed — will use Haar cascade fallback for face detection")

## Step 7 — Run preprocessing

Processes all 6,277 CMU-MOSEI segments → `z_at/*.pt` + `z_v/*.pt`.  
Other dataset manifests (tracks 1-4, MELD, MUStARD) are absent → auto-skipped.  
Resume-safe: already-cached `.pt` files are skipped on re-run.

In [ ]:
import os
os.chdir("/content/thesis")  # ensure repo root is CWD

!python scripts/preprocess_all.py --device cuda 2>&1

## Step 8 — Verify output counts

In [ ]:
from pathlib import Path

z_at_files = list(Path("data/preprocessed/features/z_at").glob("*.pt"))
z_v_files  = list(Path("data/preprocessed/features/z_v").glob("*.pt"))
wav_files  = list(Path("data/preprocessed/audio").glob("*.wav"))
txt_files  = list(Path("data/preprocessed/transcripts").glob("*.txt"))

print(f"z_at .pt files : {len(z_at_files)}")
print(f"z_v  .pt files : {len(z_v_files)}")
print(f"audio WAVs     : {len(wav_files)}")
print(f"transcripts    : {len(txt_files)}")

failed_log = Path("data/preprocessed/failed_clips.txt")
if failed_log.exists():
    failed = failed_log.read_text().strip().splitlines()
    print(f"\nFailed clips   : {len(failed)}")
    if failed:
        print("First 10 failed:", failed[:10])
else:
    print("\nNo failed_clips.txt — all succeeded or not yet written")

# Spot-check one file
if z_at_files:
    import torch
    sample = torch.load(z_at_files[0], weights_only=True)
    print(f"\nSample z_at shape : {sample.shape}  (expected torch.Size([1536]))")
if z_v_files:
    import torch
    sample = torch.load(z_v_files[0], weights_only=True)
    print(f"Sample z_v shape  : {sample.shape}  (expected torch.Size([768]))")

## Step 9 — Zip features + save to Drive

In [ ]:
import shutil, os
from pathlib import Path

features_dir = Path("/content/thesis/data/preprocessed/features")
zip_local    = "/content/mosei_features"
zip_drive    = os.path.join(DRIVE_OUTPUT_DIR, "mosei_features.zip")

print("Zipping features dir...")
shutil.make_archive(zip_local, "zip", features_dir.parent, features_dir.name)

zip_size_mb = os.path.getsize(zip_local + ".zip") / 1e6
print(f"Archive: {zip_local}.zip ({zip_size_mb:.1f} MB)")

print(f"Copying to Drive: {zip_drive} ...")
shutil.copy2(zip_local + ".zip", zip_drive)
print(f"Done. Download from Drive: {zip_drive}")

## Step 10 — (Optional) Also save failed_clips.txt + transcripts to Drive

In [ ]:
import shutil
from pathlib import Path

# Save failed clips log
failed_log = Path("/content/thesis/data/preprocessed/failed_clips.txt")
if failed_log.exists():
    dest = os.path.join(DRIVE_OUTPUT_DIR, "failed_clips.txt")
    shutil.copy2(str(failed_log), dest)
    print(f"failed_clips.txt saved to Drive: {dest}")
else:
    print("No failed_clips.txt (all succeeded)")

# Save transcripts (lightweight, useful for debugging)
transcripts_dir = Path("/content/thesis/data/preprocessed/transcripts")
if any(transcripts_dir.glob("*.txt")):
    txt_zip = "/content/mosei_transcripts"
    shutil.make_archive(txt_zip, "zip", transcripts_dir.parent, transcripts_dir.name)
    dest = os.path.join(DRIVE_OUTPUT_DIR, "mosei_transcripts.zip")
    shutil.copy2(txt_zip + ".zip", dest)
    print(f"Transcripts saved to Drive: {dest}")

## How to use the output locally

1. Download `mosei_features.zip` from Drive
2. Extract into `data/preprocessed/` — the `features/z_at/` and `features/z_v/` folders drop right in
3. The preprocessing script is resume-safe: re-running `preprocess_all.py` locally will skip all MOSEI clips already cached

```powershell
# Verify merge was successful
python -c "
from pathlib import Path
z_at = list(Path('data/preprocessed/features/z_at').glob('*.pt'))
z_v  = list(Path('data/preprocessed/features/z_v').glob('*.pt'))
print(f'z_at: {len(z_at)} files')
print(f'z_v : {len(z_v)} files')
"
```